In [1]:
import os
import copy
from PIL import Image
import torch
from PIL import ImageDraw, ImageFont
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import numpy as np

import face_recognition
from PIL import Image, PngImagePlugin, ImageDraw, ImageFont
import numpy as np
import os
import copy
import distinctipy
import dlib

In [2]:
processing_folder = r"C:\Users\Xapha\dev\bib-tagging\model_training\active_processing"
image_file = r"images\20231123-BI6I0015.jpg"

In [61]:
checkpoint = "google/owlvit-base-patch32"
model = AutoModelForZeroShotObjectDetection.from_pretrained(checkpoint)
processor = AutoProcessor.from_pretrained(checkpoint)

In [78]:
def find_people(model, processor, checkpoint, image_file):
    model = AutoModelForZeroShotObjectDetection.from_pretrained(checkpoint)
    processor = AutoProcessor.from_pretrained(checkpoint)
    text_queries=["person"]
    
    font = ImageFont.truetype("arial.ttf", 28, encoding="unic")

    im = Image.open(image_file)
    im = Image.fromarray(np.uint8(im)).convert("RGB")

    inputs = processor(text=text_queries, images=im, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)
        target_sizes = torch.tensor([im.size[::-1]])
        results = processor.post_process_object_detection(outputs, threshold=0.06, target_sizes=target_sizes)[0]

    person_ann_image = copy.deepcopy(im)
    draw = ImageDraw.Draw(person_ann_image)

    scores = results["scores"].tolist()
    labels = results["labels"].tolist()
    boxes = results["boxes"].tolist()

    for box, score, label in zip(boxes, scores, labels):
        xmin, ymin, xmax, ymax = box
        draw.rectangle((xmin, ymin, xmax, ymax), outline="red", width=4)
        draw.text((xmin, ymin), f"{text_queries[label]}: {round(score,2)}", font=font, fill="white")

    return results, person_ann_image


def box_to_images(boxes, processing_folder, image_file):

    if type(boxes) != list:
        boxes = boxes.tolist()
    im = Image.open(image_file)
    for i, box in enumerate(boxes):

        cropped_image_name = image_file.split("\\")[-1].split(".")[0] + f"_{i}.jpg"
        cropped_image_path = os.path.join(processing_folder, cropped_image_name)

        im_section = im.crop(box)
        im_section.save(cropped_image_path)

In [77]:
results, person_ann_image = find_people(processor=processor, checkpoint=checkpoint, model=model, image_file=image_file)

In [79]:
box_to_images(results['boxes'], processing_folder=processing_folder, image_file=image_file)

In [3]:
def detect_faces(image_path):
    # Even detects blurry faces. Hopefully doesn't assign numbers to them

    # Load the image
    image = face_recognition.load_image_file(image_path)

    # Find all faces in the image
    face_locations = face_recognition.face_locations(image, model="cnn", number_of_times_to_upsample=1)

    encodings = face_recognition.face_encodings(image, known_face_locations=face_locations)

    return face_locations, encodings


def draw_face_box(face_locations, image_path):
    
    im = Image.open(image_path)

    draw = ImageDraw.Draw(im)

    color_map = color_list = distinctipy.get_colors(20)
    color_list = distinctipy.get_colors(20)

    for i, face_location in enumerate(face_locations):
        color_256 = distinctipy.get_rgb256(color_list[i])
        ymin, xmax, ymax, xmin = face_location 
        draw.rectangle((xmin, ymin, xmax, ymax), outline=color_256, width=4)
        draw.text((xmax - 25, ymax - 35), f"{i}", font=font, fill=color_256)
        color_map.append(color_256)

    return im, color_map


In [9]:
number_and_face_database = [] # [(num, face_encoding), ....]

In [20]:
current_image_numbers = []

iterator = 0 # stand-in for the bib number

for image_path in os.listdir(processing_folder):
    image_path = os.path.join(processing_folder, image_path)
    face_locations, encodings = detect_faces(image_path)

    cos_sim_threshold = 0.4

    for e in encodings:
        no_matches = True
        for num, face_encoding in number_and_face_database:
            is_match = True if face_recognition.face_distance([face_encoding], e)[0] <= cos_sim_threshold else False
            if is_match:
                no_matches = False
                current_image_numbers.append(num) #  assign the image the appropriate number
                break # terminate early
        if no_matches:
            # check for a bib
            # if no bib:
                # do not add (either obscured or too far away to tell)
            # if a bib is found:
            number_and_face_database.append((iterator, e))


# So far so good!
print(len(number_and_face_database))


    

4
